# G9 — Final Preprocessing Audit

This notebook **does not** retrain or download any models. It checks the existing artifacts, adds lightweight remaining EDA, and writes `results/preprocessing_release/`. It deliberately does **not** declare the dataset frozen while the official shared stratified test set remains unresolved.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%pip -q install faiss-cpu

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
import json, hashlib, re, zipfile
import pandas as pd
import numpy as np
import faiss
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/G9_Hybrid_RAG')
DATA = ROOT/'data'
EDA = ROOT/'results/eda'
QA = ROOT/'results/qa'
RELEASE = ROOT/'results/preprocessing_release'
for d in [EDA,QA,RELEASE]: d.mkdir(parents=True,exist_ok=True)

with open(DATA/'splits/split_ids.json') as f: splits=json.load(f)
ids={s:set(map(int,splits[s])) for s in ['knowledge_base','validation','test']}
manifest=pd.read_csv(DATA/'splits/image_manifest.csv')
caps=pd.read_csv(DATA/'preprocessed_captions/kb_captions_retrieval.csv')
with open(DATA/'approved_annotations/kb_instances.json') as f: inst=json.load(f)

checks={}
checks['split_counts'] = [len(ids[s]) for s in ids]==[10000,100,100]
checks['split_disjoint'] = all(ids[a].isdisjoint(ids[b]) for a,b in combinations(ids,2))
checks['manifest_ids'] = len(manifest)==10200 and all(
    set(manifest.loc[manifest.split==s,'image_id'].astype(int))==ids[s] for s in ids)
checks['captions_five_each'] = (len(caps)==50000 and
    set(caps.image_id.astype(int))==ids['knowledge_base'] and
    caps.groupby('image_id').size().eq(5).all())
checks['kb_annotations_scoped'] = all(
    a['image_id'] in ids['knowledge_base'] for a in inst['annotations'])

qa_path=QA/'image_preprocessing_audit.csv'
image_qa=pd.read_csv(qa_path)
checks['image_preprocessing_pass']=(len(image_qa)==10200 and
    image_qa['status'].eq('PASS').all())

emb=DATA/'embeddings/clip_vit_base_patch32'
image_ids=np.load(emb/'kb_image_ids.npy')
img=np.load(emb/'kb_image_embeddings.npy',mmap_mode='r')
txt=np.load(emb/'kb_text_embeddings.npy',mmap_mode='r')
checks['image_embedding_shape']=(img.shape==(10000,512) and
    set(map(int,image_ids))==ids['knowledge_base'])
checks['text_embedding_shape']=(txt.shape==(50000,512) and
    (emb/'kb_text_metadata.csv').exists())
checks['embeddings_finite_and_normalized']=all(
    np.isfinite(block).all() and np.allclose(
        np.linalg.norm(block,axis=1),1,atol=2e-3)
    for arr in [img,txt] for pos in range(0,len(arr),2048)
    for block in [np.asarray(arr[pos:pos+2048])])

idx_dir=DATA/'indexes'
img_ix=faiss.read_index(str(idx_dir/'faiss_images.index'))
txt_ix=faiss.read_index(str(idx_dir/'faiss_captions.index'))
checks['faiss_index_counts']=(img_ix.ntotal==10000 and
    txt_ix.ntotal==50000 and img_ix.d==txt_ix.d==512)
checks['bm25_exists']=(idx_dir/'bm25.pkl').exists()

# Fast exact-duplicate screening across splits using ZIP CRC+size,
# with SHA256 confirmation only for possible matches.
archive_root=DATA/'images_archives'
name_to_split={f"{r.split}/{r.file_name}":r.split for r in manifest.itertuples()}
crc_groups=defaultdict(list)
seen_names=set()
for split in ids:
    for zp in sorted((archive_root/split).glob('part_*.zip')):
        with zipfile.ZipFile(zp) as z:
            for entry in z.infolist():
                if entry.is_dir(): continue
                assert entry.filename in name_to_split,entry.filename
                assert entry.filename not in seen_names,entry.filename
                seen_names.add(entry.filename)
                crc_groups[(entry.CRC,entry.file_size)].append((split,zp,entry.filename))
checks['archive_manifest_match']=len(seen_names)==10200 and seen_names==set(name_to_split)

def file_digest(zip_path,name):
    with zipfile.ZipFile(zip_path) as z:
        return hashlib.sha256(z.read(name)).hexdigest()

cross_split_exact=[]
for candidates in crc_groups.values():
    if len({s for s,*_ in candidates})<2: continue
    hashes=defaultdict(set)
    for split,zp,name in candidates:
        hashes[file_digest(zp,name)].add(split)
    if any(len(v)>1 for v in hashes.values()):
        cross_split_exact.append([n for _,_,n in candidates])
checks['no_exact_cross_split_duplicates']=not cross_split_exact

# Stratification diagnostic using COCO val2017 object labels.
# Diagnostic only: this does not convert a random test split into a stratified one.
val_path=DATA/'annotations/instances_val2017.json'
with open(val_path) as f: original_val=json.load(f)
val_image_ids={int(im['id']) for im in original_val['images']}
val_cats=defaultdict(set)
for ann in original_val['annotations']:
    val_cats[int(ann['image_id'])].add(int(ann['category_id']))
rows=[]
for category in original_val['categories']:
    cid=int(category['id'])
    n_all=sum(cid in val_cats[imid] for imid in val_image_ids)
    n_test=sum(cid in val_cats[imid] for imid in ids['test'])
    rows.append({'category':category['name'], 'val2017_images':n_all,
                 'test_images':n_test, 'val_prevalence':n_all/len(val_image_ids),
                 'test_prevalence':n_test/len(ids['test']),
                 'absolute_prevalence_gap':abs(n_test/len(ids['test'])-n_all/len(val_image_ids))})
pd.DataFrame(rows).sort_values('absolute_prevalence_gap',ascending=False).to_csv(
    QA/'test_stratification_diagnostic.csv',index=False)

# Add lightweight missing EDA: caption-based attributes/context
colors=['black','white','red','blue','green','yellow','brown','orange','pink','purple','gray','grey']
contexts={
    'indoor_cue': ['indoor','indoors','kitchen','bathroom','bedroom','living room'],
    'outdoor_cue': ['outdoor','outdoors','street','beach','mountain','park','field'],
    'night_cue': ['night','nighttime','evening'],
    'day_cue': ['daytime','daylight','sunny','morning']
}
cues={c:Counter() for c in ['colors','contexts']}
for row in caps.itertuples(index=False):
    text=' '+str(row.caption_clean).lower()+' '
    tokens=set(text.split())
    for color in colors:
        if color in tokens:cues['colors'][color]+=1
    for ctx,terms in contexts.items():
        if any((' '+x+' ') in text for x in terms):cues['contexts'][ctx]+=1
for name,counts in cues.items():
    pd.DataFrame(counts.most_common(),columns=['cue','caption_count']).to_csv(
        EDA/f'{name}_caption_cues.csv',index=False)

# Bounding-box proximity proxies; these are NOT semantic relations.
by_image=defaultdict(list)
for a in inst['annotations']:
    if a.get('iscrowd',0): continue
    x,y,w,h=a['bbox']
    if w>0 and h>0:by_image[a['image_id']].append((x,y,w,h))
spatial=Counter()
for boxes in by_image.values():
    for (x1,y1,w1,h1),(x2,y2,w2,h2) in combinations(boxes,2):
        spatial['pair_count']+=1
        if abs((x1+w1/2)-(x2+w2/2))>(w1+w2)/4:
            spatial['horizontal_center_separation']+=1
        if abs((y1+h1/2)-(y2+h2/2))>(h1+h2)/4:
            spatial['vertical_center_separation']+=1
        if min(x1+w1,x2+w2)>max(x1,x2) and min(y1+h1,y2+h2)>max(y1,y2):
            spatial['overlapping_bboxes']+=1
pd.DataFrame(list(spatial.items()),columns=['proxy','count']).to_csv(
    EDA/'spatial_bbox_proxy_counts.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4))
keys=[k for k in spatial if k!='pair_count']
ax.bar(keys,[spatial[k] for k in keys])
ax.tick_params(axis='x',labelrotation=20)
ax.set_ylabel('Object-pair count')
ax.set_title('Bounding-box geometry proxies (not semantic relationships)')
fig.tight_layout()
fig.savefig(EDA/'spatial_bbox_proxies.png',dpi=300,bbox_inches='tight')
plt.show()

# Freeze candidate only: professor shared/stratified test split is unresolved.
pending=['Professor-confirmed shared stratified TEST IDs (PDF says both 100 and 1000)',
         'Confirm context/attribute annotations if professor requires visually verified labels',
         'Perceptual near-duplicate detection (exact byte duplicates checked)']
summary={
    'project':'G9 Hybrid RAG COCO',
    'status':'READY_FOR_REVIEW_NOT_FROZEN',
    'checks':{k:bool(v) for k,v in checks.items()},
    'all_automated_checks_pass':all(checks.values()),
    'known_pending':pending,
    'caption_context_color_source':'caption keyword heuristics, not image-ground-truth',
    'spatial_source':'COCO bbox pair geometry, not semantic relations',
    'exact_cross_split_duplicate_groups':cross_split_exact,
    'files':{
        'split_ids':'data/splits/split_ids.json',
        'kb_captions':'data/preprocessed_captions/kb_captions_retrieval.csv',
        'image_emb':'data/embeddings/clip_vit_base_patch32/kb_image_embeddings.npy',
        'text_emb':'data/embeddings/clip_vit_base_patch32/kb_text_embeddings.npy',
        'image_index':'data/indexes/faiss_images.index',
        'text_index':'data/indexes/faiss_captions.index',
        'bm25':'data/indexes/bm25.pkl'
    }
}
(RELEASE/'preprocessing_audit.json').write_text(
    json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
small_hashes={}
for path in [DATA/'splits/split_ids.json', DATA/'splits/image_manifest.csv',
             DATA/'preprocessed_captions/kb_captions_retrieval.csv']:
    small_hashes[str(path.relative_to(ROOT))]=hashlib.sha256(path.read_bytes()).hexdigest()
(RELEASE/'small_file_sha256.json').write_text(
    json.dumps(small_hashes,indent=2),encoding='utf-8')
readme=[
    '# G9 — Preprocessing release candidate',
    'This release has not been frozen. See preprocessing_audit.json.',
    'COCO: 10,000 KB + 100 validation + 100 random test images.',
    'KB: 50,000 cleaned retrieval captions and original captions kept.',
    'Image & text: CLIP ViT-B/32, 512 dimensions, L2-normalized.',
    'Indexes: FAISS Flat inner-product + BM25.',
    'Test split is NOT yet confirmed as a shared stratified evaluation subset.',
    'Caption cue statistics are heuristics, not visual ground truth.',
    'BBox spatial proxies are geometric, not semantic object relations.',
    '', '## Automated checks',
    *[f'- {k}: {"PASS" if v else "FAIL"}' for k,v in checks.items()],
    '', '## Pending',
    *[f'- {item}' for item in pending],
    '', 'Do not change split IDs or overwrite embeddings/indexes until all teams agree on the official split.'
]
(RELEASE/'README_preprocessing.md').write_text('\n'.join(readme)+'\n',encoding='utf-8')
print('\n===== FINAL PREPROCESSING AUDIT =====')
for name,ok in checks.items():print(('✅' if ok else '❌'),name)
print('\nAutomated checks:', 'PASS' if all(checks.values()) else 'REVIEW FAILURES')
print('Release:',RELEASE/'preprocessing_audit.json')
print('Important: NOT FROZEN until official stratified test split is confirmed.')

## After running

Read `results/preprocessing_release/preprocessing_audit.json`, `README_preprocessing.md`, and `results/qa/test_stratification_diagnostic.csv`. If any automated check fails, repair it first. Before final freezing, confirm the **official shared stratified test split** with the instructor; the assignment inconsistently specifies 100 and 1,000 images. Never change IDs without regenerating split-dependent artifacts.